# Motivação sobre ML-Based Intrusion Detection

Mesmo com diferentes mecanismos de autenticação, firewall, criptografia e segurança de modo geral sendo aplicados em todas as camadas, ataques ainda são
possíveis e acontecem.
<br><br>
Sistemas de detecção de intrusão são o último recurso de segurança

*   Detectam e reportam a ocorrência ataques e atividades suspeitas quando outros mecanismos de segurança não conseguem impedi-los
*   Tais como outros mecanismos de segurança, são amplamente utilizados
    * Mas costumam apresentar vários problemas e limitações
        * Dependência de assinaturas e limitação a ataques conhecidos
        * Necessidade de analisar e rotular grandes quantidades de dados
        * Tempos de detecção longos

<br><br>
O uso de inteligência artificial pode ajudar a superar esses problemas e limitações




## Resumo comparativo de diferentes tipos de sistemas de detecção de intrusão

| **Tipo de Sistema**| **Vantagens**| **Desvantagens**|
|---|---|---|
| **IDSs baseados em regras** | - Simplicidade e facilidade de implementação<br>- Facilmente compreensíveis e auditáveis | - Manutenção intensiva devido à necessidade de atualização constante das regras<br>- Incapacidade de detectar novos ataques ou variantes desconhecidas<br>  |
| **IDSs baseados em aprendizagem supervisionada**     | - Altas taxas de detecção para ataques conhecidos | - Requerem dados maliciosos rotulados para treinamento<br>- Dificuldade em generalizar para ataques desconhecidos<br> - Tempo e custo envolvidos no treinamento do modelo<br> - Complexidade para interpretar resultados  |
| **IDSs baseados em aprendizagem não-supervisionada** | - Capacidade de detectar ataques desconhecidos e anomalias<br>- Não requerem dados maliciosos rotulados para treinamento | - Maior taxa de falsos positivos devido à natureza de detecção baseada em anomalias<br>- Tempo e custo envolvidos no treinamento do modelo<br> - Complexidade para interpretar resultados |


<font color="blue">Neste curso, vamos focar em **sistemas baseados em aprendizagem não-supervisionada**! <font>

# Bibliotecas

In [ ]:
import numpy as np
import pandas as pd
from google.colab import drive
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tqdm.notebook import tqdm
from sklearn.cluster import KMeans

from sklearn.metrics import silhouette_score
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import confusion_matrix

from tqdm.notebook import tqdm
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
RANDOM_SEED = 33
np.random.seed(RANDOM_SEED)

# Download CIC IDS 2017

O [CICIDS2017](https://www.unb.ca/cic/datasets/ids-2017.html) é um conjunto de dados que contém dados de fluxos de rede de tráfego benigno e de diferentes tipos de ciberataques.

O ambiente de teste considerado na construção deste dataset foi uma rede configurada para o atacante e uma rede separada configurada para as vítimas, contendo firewalls, roteadores, switches, servidores e estações de trabalho executando diferentes versões dos sistemas operacionais Windows e Linux.

O tráfego benigno foi gerado a partir do comportamento abstrato de 25 usuários com base em diferentes protocolos da camada de aplicação.

Network flow features (atributos dos fluxos de rede) são extraídos a partir da ferramenta CICFlowMeter8.



In [ ]:
# update gdown version
# !pip install --upgrade --no-cache-dir gdown

In [ ]:
# !wget 'http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip' -O CIC_IDS_2017.zip
!gdown '1X9s72a_9VzukVbFhGKHW1xLnyrnqu3MP' -O CIC_IDS_2017.zip

Downloading...
From (original): https://drive.google.com/uc?id=1X9s72a_9VzukVbFhGKHW1xLnyrnqu3MP
From (redirected): https://drive.google.com/uc?id=1X9s72a_9VzukVbFhGKHW1xLnyrnqu3MP&confirm=t&uuid=0f8a6f30-ace3-436e-84bd-231694fbeec4
To: /content/CIC_IDS_2017.zip
100% 235M/235M [00:05<00:00, 39.3MB/s]


In [ ]:
# !unzip MachineLearningCSV.zip
!unzip CIC_IDS_2017.zip

Archive:  CIC_IDS_2017.zip
   creating: MachineLearningCVE/
  inflating: MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv  
  inflating: MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv  


# Carregando os dados

In [ ]:
df_list = []
for file in os.listdir('MachineLearningCVE/'):
  df_aux = pd.read_csv(f'MachineLearningCVE/{file}')
  df_list.append(df_aux)
df = pd.concat(df_list, ignore_index=True)

In [ ]:
del df_list, df_aux

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max         

In [ ]:
list(df.columns)[:6]

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets']

Algumas colunas tem seus nomes iniciados com espaços ou finalizados com espaços. Vamos remover esses espaços não úteis para ajustar o nome das colunas.

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  

# Limpando os dados

É necessário limpar os dados realizando:
- Descarte de registros duplicados
- Descarte de registros com valores NaN (Not a Number)/ Null / NA (Not Available)
- Evitar registros com valores não finitos. Nesse caso, uma abordagem válida é substituirmos os mesmos pelo maior valor finito presente no dataset.

Registros duplicados

In [ ]:
df[df.duplicated()]

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
172,389,0,2,0,7,0,7,0,3.5,4.949747,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
178,389,1,2,0,7,0,7,0,3.5,4.949747,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
227,389,1,2,0,39,0,39,0,19.5,27.577164,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
895,80,45335,2,0,0,0,0,0,0.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1056,80,3,2,0,0,0,0,0,0.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830685,137,4,2,0,124,0,62,62,62.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830694,53,184,2,2,84,310,42,42,42.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830696,53,168,2,2,72,194,36,36,36.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830704,53,176,2,2,142,242,71,71,71.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [ ]:
# Descartando duplicadas
initial_len = df.shape[0]
df = df.drop_duplicates()
print(f'Tamanho inicial: {initial_len}, tamanho final {df.shape[0]} | Descartadas {initial_len - df.shape[0]} duplicadas')

Tamanho inicial: 2830743, tamanho final 2522362 | Descartadas 308381 duplicadas


Registros com valores Null/NaN/NA

In [ ]:
df.columns[df.isna().any(axis=0)]

Index(['Flow Bytes/s'], dtype='object')

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

,0
Flow Bytes/s,353


In [ ]:
# Descartando registros com valores NaN/Null/NA
initial_len = df.shape[0]
df = df.dropna()
print(f'Tamanho inicial: {initial_len}, tamanho final {df.shape[0]} | Descartados {initial_len - df.shape[0]} registros com valores NA')

Tamanho inicial: 2522362, tamanho final 2522009 | Descartados 353 registros com valores NA


In [ ]:
df = df.reset_index(drop=True)

Registros com valores não finitos

In [ ]:
df_columns_isfinite = np.isfinite(df.drop(['Label'], axis='columns')).all(axis=0)
df_columns_isfinite[df_columns_isfinite == False]

,0
Flow Bytes/s,False
Flow Packets/s,False


In [ ]:
df_rows_isfinite = np.isfinite(df.drop(['Label'], axis='columns')).all(axis=1)
inf_indexes = df_rows_isfinite[df_rows_isfinite == False].index
df.iloc[inf_indexes][['Flow Bytes/s', 'Flow Packets/s', 'Flow Duration']]

,Flow Bytes/s,Flow Packets/s,Flow Duration
164,inf,inf,0
2418,inf,inf,0
4022,inf,inf,0
14969,inf,inf,0
22211,inf,inf,0
...,...,...,...
2507126,inf,inf,0
2511105,inf,inf,0
2511877,inf,inf,0
2511898,inf,inf,0


In [ ]:
# Evitando registros com valores não finitos
max_finite_flow_packets_per_sec = df[np.isfinite(df['Flow Packets/s'])]['Flow Packets/s'].max()
max_finite_flow_bytes_per_sec = df[np.isfinite(df['Flow Bytes/s'])]['Flow Bytes/s'].max()

df.loc[df['Flow Packets/s'] == np.inf, 'Flow Packets/s'] = max_finite_flow_packets_per_sec
df.loc[df['Flow Bytes/s'] == np.inf, 'Flow Bytes/s'] = max_finite_flow_bytes_per_sec

# Mini análise exploratória - Parte 1

### Quantidade de instâncias benignas x maliciosas

In [ ]:
df['Label'] = df['Label'].replace({'Web Attack � Brute Force':'Brute Force', 'Web Attack � XSS':'XSS', 'Web Attack � Sql Injection':'Sql Injection'})

In [ ]:
df_aux = df.copy()
df_aux['Label_binary'] = df['Label'].apply(lambda label: 'Malicious' if label != 'BENIGN' else 'Benign')

fig = px.histogram(df_aux, x='Label_binary', color='Label_binary', title='Contagem de amostras por classe binária',
             color_discrete_map={'Benign':px.colors.qualitative.Plotly[0],      # Blue
                                 'Malicious':px.colors.qualitative.Plotly[1]}   # Red
             )
fig.show()

**Dados não balanceados**. Impactos:
- Dificuldade de treinar modelos supervisionados
- Dificuldade de avaliar resultados com métricas tradicionais como acurácia

### Quantidade de instâncias por tipo de ataque

Abaixo está um descritivo para os ataques do dataset:

**DoS (Denial of Service)**: Esses ataques, como "DoS Hulk", "DoS GoldenEye", "DoS Slowloris", "DoS Slowhttptest" e "DDoS" visam tornar temporariamente uma máquina ou recurso de rede indisponível, sendo diferenciados pelo protocolo e estratégia usados para causar a negação de serviço. No caso do "DDoS", várias máquinas Windows 8.1 foram usadas para enviar solicitações UDP, TCP e HTTP.

**FTP Patator" e "SSH Patator**: Usam o software Patator para adivinhar senhas por força bruta com o uso de listas de palavras.

**Web - Brute Force**: Usa força bruta em uma aplicação com listas de palavras.

**Web - Injeção de SQL**: Esse ataque explora vulnerabilidades em máquinas conectadas publicamente à Internet usando injeção SQL.

**Web - XSS (Cross-Site Scripting)**: Representa injeções de scripts em aplicativos da web, visando a execução de ações maliciosas por outros usuários do aplicativo.

**PortScan**: Realizados com a ferramenta NMap, esses ataques buscam informações sobre os serviços e portas abertas em um alvo.

**Bot**: Esse ataque tem várias possibilidades, como roubo de dados, envio de spam e acesso ao dispositivo. .

**Infiltration**: Baseado na infecção de uma máquina após um usuário abrir um arquivo malicioso.

In [ ]:
px.histogram(df_aux.query('Label != "BENIGN"'),
            x='Label',
            color='Label',
            category_orders={'Label':df_aux['Label'].value_counts().index.to_list()},
            title='Contagem de amostras por classe de ataque'
            )

Ataques menos representados

In [ ]:
N_LESS_REPRESENTED_LABELS = 5


px.histogram(df_aux[df_aux['Label'].isin(df.groupby('Label').size().sort_values(ascending=False)[(-1)*N_LESS_REPRESENTED_LABELS:].index)],
             x='Label',
             color='Label',
             title=f'Contagem de amostras dos {N_LESS_REPRESENTED_LABELS} ataques com menor representatividade')


In [ ]:
del df_aux

### Estatísticas dos dados

In [ ]:
interesting_cols = ['Flow Duration', 'Flow Bytes/s', 'Total Fwd Packets', 'Average Packet Size', 'SYN Flag Count']
df[interesting_cols].describe()

,Flow Duration,Flow Bytes/s,Total Fwd Packets,Average Packet Size,SYN Flag Count
count,2.522009e+06,2.522009e+06,2.522009e+06,2.522009e+06,2.522009e+06
mean,1.658364e+07,2.404467e+06,1.027750e+01,2.123412e+02,4.874487e-02
std,3.522618e+07,5.254864e+07,7.942294e+02,3.454504e+02,2.153342e-01
min,-1.300000e+01,-2.610000e+08,1.000000e+00,0.000000e+00,0.000000e+00
25%,2.080000e+02,1.194510e+02,2.000000e+00,9.000000e+00,0.000000e+00
50%,5.058700e+04,3.722028e+03,2.000000e+00,8.075000e+01,0.000000e+00
75%,5.330376e+06,1.079162e+05,6.000000e+00,1.796923e+02,0.000000e+00
max,1.200000e+08,2.071000e+09,2.197590e+05,3.893333e+03,1.000000e+00


# Dividindo dados nos conjuntos de treino, validação e teste

**Conjunto de treino**

Para a detecção de anomalias, vamos usar somente os dados que representam o tráfego benigno para o conjunto de treino. Dessa forma, os algoritmos de clustering vão ser capazes de identificar padrões e desvios em relação ao comportamento normal (benigno) dos dados.

**Conjuntos de validação e teste**

Porém, devem ser incluídos dados que representam o tráfego maliciosos nos conjuntos de validação e teste. Esses dados maliciosos no conjunto de validação são importantes para que possamos definir um *threshold* para que seja possível detectar anomalias. Além disso, os dados maliciosos também precisam ser incluídos no conjunto de teste para que possamos avaliar o desempenho do nosso modelo.

In [ ]:
df_train = df.query('Label == "BENIGN"').sample(frac=0.6, random_state=RANDOM_SEED)
df_val_test = df.drop(df_train.index)

df_train = df_train.reset_index(drop=True)
df_val_test = df_val_test.reset_index(drop=True)

X_train = df_train.drop('Label', axis='columns')

In [ ]:
X_val, X_test, classes_val, classes_test = train_test_split(df_val_test.drop('Label', axis='columns'), df_val_test['Label'], test_size=0.65, stratify=df_val_test['Label'], random_state=RANDOM_SEED)

X_val, X_test = X_val.reset_index(drop=True), X_test.reset_index(drop=True)
classes_val, classes_test =  classes_val.reset_index(drop=True), classes_test.reset_index(drop=True)

y_val, y_test = classes_val.apply(lambda c: 0 if c == 'BENIGN' else 1), classes_test.apply(lambda c: 0 if c == 'BENIGN' else 1)

# Mini Análise Exploratória - Parte 2

In [ ]:
df_val = X_val.copy()
df_val['Label'] = classes_val
df_val['isAttack'] = df_val['Label'].apply(lambda l: 0 if l == 'BENIGN' else 1)

In [ ]:
px.box(df_val, x='Destination Port', y='Label', color='Label', title='Boxplot da ocorrência de portas por categoria')

Note que a maioria dos ataques foca em apenas uma ou poucas portas. Além disso o tráfego benigno ocorre principalmente com portas de destino bem conhecidas (0 a 1023). Por outro lado, o ataque de PortScan, por ser característico do mesmo verficar informações sobre uma grande quantidade de portas de um host, abrange maior variedade de portas.

In [ ]:
px.box(df_val, x='Active Max', y='Label', color='Label', title='Boxplot do tempo máximo que um fluxo esteve ativo antes de se tornar ocioso')

Podemos notar que alguns ataques possuem a mediana da feature `Active Max`, que representa o tempo máximo que um fluxo esteve ativo antes de se tornar ocioso, significativamente mais alta do que do comportamento benigno, de forma que o último possui o valor 0 como mediana para essa feature.

Alguns ataques com valores significativamente diferentes para essa feature foram: `DoS Slowloris`, `DoS Slowhttptest` e `Infiltration`,

# Analisando correlação entre features

## Explorando quais as features com maiores correlações para cada ataque

In [ ]:
corr_matrix_list = []
relevant_attacks = ['DDoS', 'DoS Hulk', 'DoS GoldenEye', 'PortScan', 'SSH-Patator', 'XSS', 'Infiltration']
df_val_benign = df_val.query('Label == "BENIGN"')

for attack in relevant_attacks:
    df_val_attack = df_val.query(f'Label == "{attack}"')
    df_val_attack_benign = pd.concat([df_val_attack, df_val_benign], ignore_index=True)
    corr_matrix_list.append(df_val_attack_benign.drop(['Label'], axis='columns').corr())

In [ ]:
N = len(corr_matrix_list)

fig = make_subplots(rows=N, cols=1, subplot_titles=[f'{relevant_attacks[i]}' for i in range(N)])

# Adicionando um bar plot a cada subplot
for i in range(N):
    #curr_corr_series = corr_matrix_list[i]['isAttack'].sort_values(ascending=False)[1:11]

    curr_top_10_idx = corr_matrix_list[i]['isAttack'].abs().nlargest(11).index
    curr_corr_series = corr_matrix_list[i]['isAttack'][curr_top_10_idx][1:]

    fig.add_trace(
        go.Bar(x=curr_corr_series.index, y=curr_corr_series, name=f'{relevant_attacks[i]}'),
        row=i+1, col=1
    )
    fig.update_xaxes(tickangle=5, row=i+1, col=1)


# Atualizando o layout
fig.update_layout(height=1000, width=1600, title_text="Top 10 features com maior correlação em módulo por categoria de ataque")

# Exibindo o gráfico
fig.show()

Note que alguns ataques possuem features com correlação relativamente alta. Por exemplo, a feature `Bwd Packet Lenght Mean` tem uma correlação de 0,83 para o ataque `DoS Hulk` e uma correlação de 0,66 para o ataque `DDoS`

<br>

Por outro lado, alguns ataques como `XSS`, `Infiltration` e `SSH-Patator` apresentam baixa correlação para todas as features. Isso pode indicar maior dificuldade para detecção dos mesmos utilizando as features atuais.

<br>

De fato, nem sempre fluxos de rede são o tipo de log ideal para detectar ataques porque eles capturam apenas informações básicas sobre o tráfego, como endereços IP e portas, sem detalhes sobre o conteúdo das comunicações. Por exemplo, para detecção de ataques em aplicações web, pode ser mais indicado o uso de logs associados ao tráfego HTTP/HTTPS do que fluxos de rede, por fornecerem informações mais detalhados sobre os pontos que são afetados durante esses tipos de ataque, como parâmetros das requisições.

## Removendo features redundantes

**Por que remover features?**

Vamos descartar features com alta correlação evitando passar informações redundantes ao modelo. Dessa forma, conseguiremos obter um modelo mais simples e com menor custo computacional.

In [ ]:
def get_highly_correlated_features(correlation_matrix, threshold):
  correlated_pairs = []
  for i in range(len(correlation_matrix.columns)):
    for j in range(i):
      if abs(correlation_matrix.iloc[i, j]) > threshold:
        pair = (correlation_matrix.columns[i], correlation_matrix.columns[j])
        coefficient = correlation_matrix.iloc[i, j]
        correlated_pairs.append((pair, coefficient))
  return sorted(correlated_pairs, key= lambda pair: pair[1], reverse=True)

In [ ]:
corr_matrix = X_train.corr().abs()
correlation_list = get_highly_correlated_features(corr_matrix, 0.95)

In [ ]:
correlation_list[:10]

[(('SYN Flag Count', 'Fwd PSH Flags'), np.float64(1.0)),
 (('CWE Flag Count', 'Fwd URG Flags'), np.float64(1.0)),
 (('Avg Fwd Segment Size', 'Fwd Packet Length Mean'), np.float64(1.0)),
 (('Fwd Header Length.1', 'Fwd Header Length'), np.float64(1.0)),
 (('Subflow Fwd Packets', 'Total Fwd Packets'), np.float64(1.0)),
 (('Subflow Bwd Packets', 'Total Backward Packets'), np.float64(1.0)),
 (('Avg Bwd Segment Size', 'Bwd Packet Length Mean'),
  np.float64(0.9999999999999999)),
 (('Subflow Bwd Bytes', 'Total Length of Bwd Packets'),
  np.float64(0.9999998720308314)),
 (('Subflow Fwd Bytes', 'Total Length of Fwd Packets'),
  np.float64(0.9999994944808139)),
 (('Total Backward Packets', 'Total Fwd Packets'),
  np.float64(0.9993836166392885))]

In [ ]:
# Drop high correlated features in correlation list

f2drop = []
for feature_pair, _ in correlation_list:
  if feature_pair[0] not in f2drop and feature_pair[1] not in f2drop:
    f2drop.append(feature_pair[1])

In [ ]:
f2drop

['Fwd PSH Flags',
 'Fwd URG Flags',
 'Fwd Packet Length Mean',
 'Fwd Header Length',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Bwd Packet Length Mean',
 'Total Length of Bwd Packets',
 'Total Length of Fwd Packets',
 'Subflow Fwd Packets',
 'Flow Duration',
 'RST Flag Count',
 'Subflow Bwd Packets',
 'Packet Length Mean',
 'Flow IAT Max',
 'Idle Mean',
 'Fwd IAT Total',
 'Max Packet Length',
 'Fwd Packet Length Max',
 'Bwd IAT Max',
 'Bwd IAT Mean',
 'Fwd IAT Max',
 'Fwd IAT Mean',
 'Idle Max']

A feature "Destination Port", embora tenha relevância para detecção de alguns ataques como Brute Force SSH, Brute Force FTP, PortScan, etc, está representada com valores inteiros. Porém, isso traz ao modelo uma falsa relação de grandeza entre número de portas, como que a porta 44720 > porta 80, o que não apresenta muito sentido quando se trata de uma porta de destino de um fluxo de rede.

In [ ]:
f2drop.append('Destination Port')

In [ ]:
X_train = X_train.drop(f2drop, axis='columns')
X_val = X_val.drop(f2drop, axis='columns')
X_test = X_test.drop(f2drop, axis='columns')

# Normalizando os dados

É importante normalizar os dados para lidar com diferentes escalas, sensibilidades a escalas e até mesmo melhorar o desempenho da convergência dos algoritmos.

Caso não seja realizada a normalização, um valor de 10000 para uma feature como "Flow Bytes/s" terá impacto similar ao modelo quanto um valor de 10000 para uma feature como "Flow Packets/s". Isso é prejudicial, pois o impacto desse valor para as duas features deve ser tratado de forma distinta, já que as mesmas têm escalas e sensibilidades também distintas.

In [ ]:
# Usando MinMax Scaler dessa vez para que a rede neural seja capaz de gerar saídas no intervalo numérico da função sigmóide

minmax_scaler = MinMaxScaler()
minmax_scaler = minmax_scaler.fit(X_train)

norm_X_train = minmax_scaler.transform(X_train)
norm_X_val = minmax_scaler.transform(X_val)
norm_X_test = minmax_scaler.transform(X_test)